# Survivor Data Set: An Exploration
## Project Overview

We seek to understand the underlying themes in the reality TV gameshow "Survivor". The show is currently airing it's 50th season, but the gameplay we see in the later seasons is very different from the early seasons of the show. Survivor is a gameshow about physical challenges and mental strength, but fundamentally it is a show about relationships with others. In a game where the goal is to "Outwit, Outplay, Outlast", what does this actually translate to? In order to win you need the votes of the jury, which is made up of the players you had a hand in voting off of the tribe.

The thing that makes Survivor so interesting is how different every season is, and how the game is always evolving. In early seasons, loyalty and morals were valued, and the gameplay was relatively simple. As time went on, players got more conniving, and blindsides and backstabbing became the norm. But what is most interesting is that as the gameplay changed, so did the mentality of the players. Playing fair isn't enough to win anymore. 

Through this data we hope to find answers to some of these questions:
- What does it take Outwit, Outplay, and Outlast?
- What decisions to winners make? 
- What decisions do losers make?
- How have the decisions made by winners and losers changed as the game has evolved?
- Are there trends that predict performance?
- Are there qualities that predict performance?
- Have there been changes in these trends/qualities over time?

## Data Description and Source
The original survivoR dataset is in R, [original data set](https://github.com/doehm/survivoR/tree/master/data). We will be using a modified version of this dataset that has been converted to CSV files found here: [Survivor Data Set](https://github.com/rfordatascience/tidytuesday/tree/master/data/2021/2021-06-01)

The original dataset has 23 R data files, while the dataset that's been converted to CSV only has 5 of these. 

The 5 csv files are:
- Summary
- Challanges
- Castaways
- Viewers
- Jury Votes


# Initial CSV Data
The following cells provide context for relavance, size, and column descriptions for each of the CSV files.

In [ ]:
import pandas as pd

# Each table is available as a CSV from the TidyTuesday GitHub mirror
base_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-06-01/"

# Load the main tables
summary_df     = pd.read_csv(base_url + "summary.csv")
castaways_df   = pd.read_csv(base_url + "castaways.csv")
viewers_df     = pd.read_csv(base_url + "viewers.csv")
challenges_df  = pd.read_csv(base_url + "challenges.csv")

print(f"Summary rows: {len(summary_df)}")
print(f"Castaway rows: {len(castaways_df)}")
print(f"Viewer rows: {len(viewers_df)}")
print(f"Challenges rows: {len(challenges_df)}")




## Summary CSV:
- Relevance: Primary data set for information on seasons, winners, dates. Usuful for determining winners, and trends over times
- Size: 
- Column Descriptions: Season Name, Season, Location, Country, Tribe Setup, Full Name, Winner, Runner Ups, Final Vote, Time Slot, Premiered, 

## Castaways Dataframe

- Relevance: Contains information on all of the contestants from all 40 seasons. This gives us the details about who the players are, and can be helpful in finding trends of winners.
- Size: 744 rows, 18 columns
- Columns: season_name, season, full_name, castaway, age, city, state, personality_type, day, order, result, jury_status, original_tribe, swapped_tribe, swapped_tribe2, merged_tribe, total_votes_received, immunity_idols_won

The following information is used to determine the cleaning steps that are needed.

In [ ]:
# Shape and column names
print("SHAPE")
print(castaways_df.shape)

print("\nCOLUMNS")
print(castaways_df.columns.tolist())

# Data types
print("\nDATA TYPES")
print(castaways_df.dtypes)

# Missing values
print("\nMISSING VALUES")
print(castaways_df.isnull().sum())

# Basic stats
print("\nBASIC STATS")
castaways_df.describe()
print("\nINFO")
castaways_df.info()

### Data Cleaning: Fixing Null values

Based on the above cell we find the following columns have null values:
| Column Name | Null Count |
|--------|-----------|
| personality_type | 3 |
| jury_status | 405 |
| original_tribe | 2 |
| swapped_tribe | 284 |
| swapped_tribe2 | 683 |
| merged_tribe | 300 |

In [ ]:
#first we create a copy of the dataframe for editing
clean_castaways_df = castaways_df.copy()

#### personality_type
First we will fix the missing personality types. Since we are only missing three, we could just drop these rows, but we may want other information on these players, so we will fill them with 'UNKNOWN' instead.

In [ ]:
print(clean_castaways_df['personality_type'].unique())

In [ ]:
clean_castaways_df['personality_type'] = clean_castaways_df['personality_type'].fillna('UNKNOWN')
print(clean_castaways_df['personality_type'].info())

#### jury_status

In [ ]:
# See all unique values in the column
print(castaways_df['jury_status'].unique())

After viewing this we can understand that this column tells us what member of the jury a castaway is. The players eliminated earlier in the season don't make it on the jury, which explains why there are so many null values. After some consideration we will replace all null values with "non jury member" to follow the naming convention. 

In [ ]:
clean_castaways_df['jury_status'] = clean_castaways_df['jury_status'].fillna('non jury member')
clean_castaways_df['jury_status'].info()

#### original_tribe
Since there are only two null values in this column, we will investigate which two castaways have the null values to determine if we should drop the rows or replace the null with a value.

In [ ]:
print(castaways_df[castaways_df['original_tribe'].isnull()])

After some googling I found this explanation from a Reddit post: "For those of you who don’t know, in the very first episode of Palau (Season 10), Jonathan Libby and Wanda Shirk were eliminated before tribes were even officially formed." Based on this we can conclude that we can drop these rows from the dataframe since they are inconsequential to understanding themes in the show. 

In [ ]:
clean_castaways_df = clean_castaways_df.dropna(subset=['original_tribe'])
clean_castaways_df.info()

#### swapped_tribe, swapped_tribe2
We will handle swapped_tribe and swapped_tribe2 together since they contain the same type of information, but some castaways swap tribes once, twice, or never.


In [ ]:
print(clean_castaways_df['swapped_tribe'].unique())
print(clean_castaways_df['swapped_tribe2'].unique())

From this output and some general knowledge, we can understand that these columns tell us what tribe a castaway switched to. However, not all players switch tribes, and especially most players don't switch tribes twice, but they are still important to the story the data is telling us. We will fill these with "Not Applicable".

In [ ]:
swap_columns = ['swapped_tribe', 'swapped_tribe2']
clean_castaways_df[swap_columns] = clean_castaways_df[swap_columns].fillna('Not Applicable')

clean_castaways_df[swap_columns].info()

#### merged_tribe

In [ ]:
print(clean_castaways_df['merged_tribe'].unique())
print(f"Number of merged tribe names: {clean_castaways_df['merged_tribe'].nunique()}")

After viewing this list of the merged tribe names we now understand that this column represents the name of the merged tribe a castaway was in. Part way through the season all of the tribes get merged into one tribe, and they create a new name for the tribe. Since around half of the players each season get eliminated before the merge, said players have null values. We will fill this with 'Not Applicable' since we still want the data on the players who don't make it to the merge.

In [ ]:
clean_castaways_df['merged_tribe'] = clean_castaways_df['merged_tribe'].fillna('Not Applicable')
clean_castaways_df['merged_tribe'].info()

#### The data for the Castaways dataframe is now cleaned.

In [ ]:
clean_castaways_df.info()